In [1]:
import numpy as np
import pandas as pd
import os
import random

# Create LOSO split

In [3]:
def generate_leave_n_subject_splits(
    subject_ids,
    folds = None,
    is_test_val=False,
    out_dir="SWELL_LOSO_splits",
    shuffle=True,
    seed=42,
):
    """
    Generate multiple CSV split files for leave-N-subject-out.

    Each CSV:
      - includes all subjects (each subject appears exactly once in {train, val, test})
      - has columns: subject_id, split

    Parameters
    ----------
    subject_ids : list of str
        e.g. ["S2", "S3", "S4", ...]
    N_test : int
        Number of subjects in the test split (leave-N-subject-out).
    M_val : int
        Number of subjects in the validation split (0 means no val).
    out_dir : str
        Directory to save CSV files.
    shuffle : bool
        Whether to shuffle the subject_ids before building splits.
    seed : int
        Random seed for reproducibility when shuffle=True.
    """
    os.makedirs(out_dir, exist_ok=True)

    subject_ids = list(subject_ids)
    num_subjects = len(subject_ids)

    # ---- sanity checks ----
    N_test = M_val = int(num_subjects / folds) if folds else 1
    if folds is None:
        folds = num_subjects // N_test
    assert folds > 0, "Number of folds must be positive."
    # Optional shuffle for randomness
    if shuffle:
        random.seed(seed)
        random.shuffle(subject_ids)
    else:
        subject_ids.sort()

    print(f"Subjects used (order): {subject_ids}")


    for k in range(folds):
        # ---- choose test subjects for this fold ----
        start = k * N_test
        end = start + N_test
        test_subjects = subject_ids[start:end]

        # remaining subjects are candidates for train/val
        remaining = [s for s in subject_ids if s not in test_subjects]

        # choose val subjects from remaining (first M_val)
        if M_val > 0 and not is_test_val:
            val_subjects = remaining[:M_val]
            train_subjects = remaining[M_val:]
        elif M_val > 0 and is_test_val:
            val_subjects = test_subjects[:M_val]
            train_subjects = remaining
        else:
            val_subjects = []
            train_subjects = remaining

        # ---- build rows ----
        rows = []
        for s in train_subjects:
            rows.append({"subject_id": s, "split": "train"})
        for s in val_subjects:
            rows.append({"subject_id": s, "split": "val"})
        for s in test_subjects:
            rows.append({"subject_id": s, "split": "test"})

        df = pd.DataFrame(rows)

        # e.g. Fold/fold_00.csv, Fold/fold_01.csv, ...
        out_path = os.path.join(out_dir, f"fold_{k:02d}.csv")
        df.to_csv(out_path, index=False)

        print(
            f"[fold_{k:02d}.csv] "
            f"train={len(train_subjects)}, val={len(val_subjects)}, test={len(test_subjects)}"
        )
        print(f"  train: {train_subjects}")
        if val_subjects:
            print(f"  val:   {val_subjects}")
        print(f"  test:  {test_subjects}")


In [4]:
subject_ids = [ f for f in os.listdir("/home/s223149341/SSL-invariance-Subject_Project_model/data/SWELL/SWELL_1280_320_2label") if f.startswith("s")]
generate_leave_n_subject_splits(
    subject_ids,
    is_test_val=True,
    out_dir="SWELL_LOSO",
    shuffle=False,   # set True if you want random order
)

Subjects used (order): ['s10_phsyio.pkl', 's11_phsyio.pkl', 's12_phsyio.pkl', 's13_phsyio.pkl', 's14_phsyio.pkl', 's15_phsyio.pkl', 's16_phsyio.pkl', 's17_phsyio.pkl', 's18_phsyio.pkl', 's19_phsyio.pkl', 's1_phsyio.pkl', 's20_phsyio.pkl', 's21_phsyio.pkl', 's22_phsyio.pkl', 's23_phsyio.pkl', 's24_phsyio.pkl', 's25_phsyio.pkl', 's2_phsyio.pkl', 's3_phsyio.pkl', 's4_phsyio.pkl', 's5_phsyio.pkl', 's6_phsyio.pkl', 's7_phsyio.pkl', 's8_phsyio.pkl', 's9_phsyio.pkl']
[fold_00.csv] train=24, val=1, test=1
  train: ['s11_phsyio.pkl', 's12_phsyio.pkl', 's13_phsyio.pkl', 's14_phsyio.pkl', 's15_phsyio.pkl', 's16_phsyio.pkl', 's17_phsyio.pkl', 's18_phsyio.pkl', 's19_phsyio.pkl', 's1_phsyio.pkl', 's20_phsyio.pkl', 's21_phsyio.pkl', 's22_phsyio.pkl', 's23_phsyio.pkl', 's24_phsyio.pkl', 's25_phsyio.pkl', 's2_phsyio.pkl', 's3_phsyio.pkl', 's4_phsyio.pkl', 's5_phsyio.pkl', 's6_phsyio.pkl', 's7_phsyio.pkl', 's8_phsyio.pkl', 's9_phsyio.pkl']
  val:   ['s10_phsyio.pkl']
  test:  ['s10_phsyio.pkl']
[fold_01

In [5]:
subject_ids = [ f for f in os.listdir("/home/s223149341/SSL-invariance-Subject_Project_model/data/verbio_1280_64") ]
generate_leave_n_subject_splits(
    subject_ids,
    folds=5,
    is_test_val=True,
    out_dir="VERBIO_L5SO",
    shuffle=False,   # set True if you want random order
)

Subjects used (order): ['P001', 'P003', 'P004', 'P005', 'P006', 'P007', 'P008', 'P009', 'P011', 'P012', 'P013', 'P014', 'P016', 'P017', 'P018', 'P020', 'P021', 'P023', 'P026', 'P027', 'P031', 'P032', 'P035', 'P037', 'P038', 'P039', 'P040', 'P041', 'P042', 'P043', 'P044', 'P045', 'P046', 'P047', 'P048', 'P049', 'P050', 'P051', 'P052', 'P053', 'P056', 'P057', 'P058', 'P060', 'P061', 'P062']
[fold_00.csv] train=37, val=9, test=9
  train: ['P012', 'P013', 'P014', 'P016', 'P017', 'P018', 'P020', 'P021', 'P023', 'P026', 'P027', 'P031', 'P032', 'P035', 'P037', 'P038', 'P039', 'P040', 'P041', 'P042', 'P043', 'P044', 'P045', 'P046', 'P047', 'P048', 'P049', 'P050', 'P051', 'P052', 'P053', 'P056', 'P057', 'P058', 'P060', 'P061', 'P062']
  val:   ['P001', 'P003', 'P004', 'P005', 'P006', 'P007', 'P008', 'P009', 'P011']
  test:  ['P001', 'P003', 'P004', 'P005', 'P006', 'P007', 'P008', 'P009', 'P011']
[fold_01.csv] train=37, val=9, test=9
  train: ['P001', 'P003', 'P004', 'P005', 'P006', 'P007', 'P00

In [2]:

def generate_cross_splits(
    train_set_path,
    test_val_set_path,
    train_is_val,
    num_fold=None,
    out_dir="SWELL_LOSO_splits",
    shuffle=True,
    seed=42,
):
    #Create the output folder
    os.makedirs(out_dir, exist_ok=True)
    
    #Read the dataset path
    train_subject = [f for f in os.listdir(train_set_path)]
    test_val_subject = [f for f in os.listdir(test_val_set_path)]
    num_subject = int(len(test_val_subject)/num_fold)
    
    # Optional shuffle for randomness
    if shuffle:
        random.seed(seed)
        random.shuffle(test_val_subject)
    else:
        test_val_subject.sort()

    print(f"Subjects used (order): {test_val_subject}")

    
    # number of folds based on the test set
    print(f"Based on the number of fold we will split the dataset, each fold will have {num_subject}")
    if len(test_val_subject) % num_fold != 0:
        raise(f"The number of subject is not completely divdied")
     
    #Fix the train set
    rows = []
    for s in train_subject:
        rows.append({"subject_id": s, "split": "train"})
    print(f"  train: {train_subject}")
    if train_is_val:
        for s in train_subject:
            rows.append({"subject_id": s, "split": "val"})
    
    fold = 0
    for i in range(0, len(test_val_subject), num_subject):
        inner_rows = rows.copy()

        test_subjects = test_val_subject[i:i+num_subject]
        if not train_is_val:
            for s in test_subjects:
                inner_rows.append({"subject_id": s, "split": "val"})
        for s in test_subjects:
            inner_rows.append({"subject_id": s, "split": "test"})

        df = pd.DataFrame(inner_rows)

        # e.g. Fold/fold_00.csv, Fold/fold_01.csv, ...
        out_path = os.path.join(out_dir, f"fold_{fold:02d}.csv")
        df.to_csv(out_path, index=False)
        #Update the folde
        fold += 1
        print(
            f"[fold_{fold:02d}.csv] "
            f"train={len(train_subject)}, val={len(test_subjects)}, test={len(test_subjects)}"
        )
        print(f"Val and test:  {test_subjects}")


In [3]:
WESAD_PATH = '/home/s223149341/SSL-invariance-Subject_Project_model/data/WESAD/wesad_10_05_no_standardize'
SWELL_PATH = '/home/s223149341/SSL-invariance-Subject_Project_model/data/SWELL/SWELL_1280_320'

In [5]:

generate_cross_splits(
    train_set_path = WESAD_PATH,
    test_val_set_path = SWELL_PATH,
    num_fold=1,
    train_is_val= False,
    out_dir="WESAD_SWELL_False",
    shuffle=True,
    seed=42
)

Subjects used (order): ['s3_phsyio.pkl', 's15_phsyio.pkl', 's11_phsyio.pkl', 's19_phsyio.pkl', 's20_phsyio.pkl', 's14_phsyio.pkl', 's9_phsyio.pkl', 's24_phsyio.pkl', 's18_phsyio.pkl', 's17_phsyio.pkl', 's22_phsyio.pkl', 's2_phsyio.pkl', 's25_phsyio.pkl', 's7_phsyio.pkl', 's6_phsyio.pkl', 's16_phsyio.pkl', 's21_phsyio.pkl', 's4_phsyio.pkl', 's13_phsyio.pkl', 's10_phsyio.pkl', 's23_phsyio.pkl', 's12_phsyio.pkl', 's5_phsyio.pkl', 's8_phsyio.pkl', 's1_phsyio.pkl']
Based on the number of fold we will split the dataset, each fold will have 25
  train: ['S15', 'S6', 'S7', 'S16', 'S5', 'S8', 'S9', 'S4', 'S10', 'S17', 'S2', 'S3', 'S13', 'S11', 'S14']
[fold_01.csv] train=15, val=25, test=25
Val and test:  ['s3_phsyio.pkl', 's15_phsyio.pkl', 's11_phsyio.pkl', 's19_phsyio.pkl', 's20_phsyio.pkl', 's14_phsyio.pkl', 's9_phsyio.pkl', 's24_phsyio.pkl', 's18_phsyio.pkl', 's17_phsyio.pkl', 's22_phsyio.pkl', 's2_phsyio.pkl', 's25_phsyio.pkl', 's7_phsyio.pkl', 's6_phsyio.pkl', 's16_phsyio.pkl', 's21_phsyio

In [5]:

generate_cross_splits(
    train_set_path = WESAD_PATH,
    test_val_set_path = SWELL_PATH,
    num_fold=1,
    out_dir="WESAD_SWELL_Full",
    shuffle=True,
    seed=42
)

Subjects used (order): ['s3_phsyio.pkl', 's15_phsyio.pkl', 's11_phsyio.pkl', 's19_phsyio.pkl', 's20_phsyio.pkl', 's14_phsyio.pkl', 's9_phsyio.pkl', 's24_phsyio.pkl', 's18_phsyio.pkl', 's17_phsyio.pkl', 's22_phsyio.pkl', 's2_phsyio.pkl', 's25_phsyio.pkl', 's7_phsyio.pkl', 's6_phsyio.pkl', 's16_phsyio.pkl', 's21_phsyio.pkl', 's4_phsyio.pkl', 's13_phsyio.pkl', 's10_phsyio.pkl', 's23_phsyio.pkl', 's12_phsyio.pkl', 's5_phsyio.pkl', 's8_phsyio.pkl', 's1_phsyio.pkl']
Based on the number of fold we will split the dataset, each fold will have 25
  train: ['S15', 'S6', 'S7', 'S16', 'S5', 'S8', 'S9', 'S4', 'S10', 'S17', 'S2', 'S3', 'S13', 'S11', 'S14']
[fold_01.csv] train=15, val=25, test=25
Val and test:  ['s3_phsyio.pkl', 's15_phsyio.pkl', 's11_phsyio.pkl', 's19_phsyio.pkl', 's20_phsyio.pkl', 's14_phsyio.pkl', 's9_phsyio.pkl', 's24_phsyio.pkl', 's18_phsyio.pkl', 's17_phsyio.pkl', 's22_phsyio.pkl', 's2_phsyio.pkl', 's25_phsyio.pkl', 's7_phsyio.pkl', 's6_phsyio.pkl', 's16_phsyio.pkl', 's21_phsyio

In [1]:

def generate_train_tests(
    train_set_path,
    train_split: float = 0.8,
    out_dir="SWELL_LOSO_splits",
    shuffle=True,
    seed=42,
):
    #Create the output folder
    os.makedirs(out_dir, exist_ok=True)
    
    #Read the dataset path
    train_subject = [f for f in os.listdir(train_set_path)]
    
    # Optional shuffle for randomness
    if shuffle:
        random.seed(seed)
        random.shuffle(train_subject)
    else:
        train_subject.sort()
    print(f"Subjects used (order): {train_subject}")

    num_trains = int(len(train_subject) * train_split)   
    train_subjects = train_subject[:num_trains]
    test_subjects = train_subject[num_trains:len(train_subject)]

    #Fix the train set
    rows = []
    for s in train_subjects:
        rows.append({"subject_id": s, "split": "train"})
    for s in test_subjects:
        rows.append({"subject_id": s, "split": "val"})
    for s in test_subjects:
        rows.append({"subject_id": s, "split": "test"})

    df = pd.DataFrame(rows)
    out_path = os.path.join(out_dir, f"train_test.csv")
    df.to_csv(out_path, index=False)


In [7]:
generate_train_tests(WESAD_PATH, 0.8,
    out_dir="WESAD_80_split",
    shuffle=True,
    seed=42)

Subjects used (order): ['S10', 'S11', 'S4', 'S9', 'S14', 'S13', 'S8', 'S7', 'S17', 'S16', 'S5', 'S3', 'S15', 'S6', 'S2']
